# Q-ErrorID: hybrid QML reconstruction

This notebook inspects the trained one- and two-qubit generator regressors. The QNN uses angle data re-uploading, ring entanglement, Pauli-Z measurements and a small physical-output head. It is a hardware-deployment demonstrator; no quantum-advantage claim is made.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from q_error_id.models import (
    HybridQNN,
    evaluate_model,
    export_qiskit_inference,
    load_model_dataset,
    recommend_strategy,
    vector_to_parameters,
)

manifest = json.loads((ROOT / 'artifacts/models/model_manifest.json').read_text())
manifest['claims']

## 1. Load Agent 1 diagnostics

The NPZ contract contains exact, readout-corrupted and finite-shot features, physical generator targets, PTMs and normalized Choi states. For the two-qubit model, local amplitude-damping rates are known nuisance values supplied by the prior one-qubit calibration.

In [ ]:
datasets = {}
for family in ('1q', '2q'):
    path = ROOT / f'artifacts/datasets/{family}_mixed_channel_test.npz'
    datasets[family] = load_model_dataset(path)
    data = datasets[family]
    print(family, 'samples=', data.size, 'features=', data.X.shape[1], 'outputs=', data.spec.names)

## 2. Inspect the trained QNN

The classical normalizer feeds a fixed feature compressor. Every quantum layer applies feature-dependent RY/RZ gates, trainable offsets and a CNOT ring. Single-Z and neighboring-ZZ expectations feed the physical head.

In [ ]:
qnn = {family: HybridQNN.load(ROOT / f'artifacts/models/qnn_{family}.pt') for family in ('1q', '2q')}
for family, model in qnn.items():
    print('\n', family)
    print(json.dumps(model.architecture(), indent=2))

## 3. Reconstruct physical channels

Evaluation rebuilds every predicted channel with Agent 1's channel builder. It reports parameter errors, Choi-state fidelity, relative PTM error and expectations on diagnostic settings that were not used as model inputs.

In [ ]:
evaluations = {}
for family in ('1q', '2q'):
    data = datasets[family]
    result = evaluate_model(
        qnn[family], data.X, data.y, data.known_kappa, data.spec,
        metadata=data.metadata,
    )
    evaluations[family] = result
    print(family, pd.Series(result.summary).to_string(), '\n')

In [ ]:
family = '1q'
data = datasets[family]
prediction = evaluations[family].predictions
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for index, axis in enumerate(axes.flat):
    if index >= data.spec.n_outputs:
        axis.axis('off'); continue
    axis.scatter(data.y[:, index], prediction[:, index], s=16, alpha=.7)
    low = min(data.y[:, index].min(), prediction[:, index].min())
    high = max(data.y[:, index].max(), prediction[:, index].max())
    axis.plot([low, high], [low, high], 'k--')
    axis.set_title(data.spec.names[index])
plt.tight_layout();

## 4. Finite-shot robustness

The exact model and the randomized-shot-augmentation model are evaluated separately at 4096, 1024 and 256 shots.

In [ ]:
shot_table = pd.read_csv(ROOT / 'results/models/shot_robustness.csv')
display(shot_table)
for family, group in shot_table.groupby('family'):
    pivot = group.pivot(index='feature_regime', columns='model', values='parameter_mae')
    pivot.loc[['exact', '4096-shot', '1024-shot', '256-shot']].plot(marker='o', title=family)
    plt.ylabel('parameter MAE'); plt.xticks(rotation=20); plt.show()

## 5. Fair classical comparison

The MLP hidden width is chosen to be close to the QNN plus classical-head parameter count. Ridge is retained as a strong linear diagnostic baseline. Better classical performance must be reported honestly.

In [ ]:
comparison = pd.read_csv(ROOT / 'results/models/model_comparison.csv')
display(comparison)

## 6. Coherent correction and strategy label

Only the Hamiltonian estimate is inverted. The dissipative component remains and is reported separately.

In [ ]:
family = '1q'
data = datasets[family]
predicted_parameters = vector_to_parameters(evaluations[family].predictions[0], data.spec)
print('recommended strategy:', recommend_strategy(predicted_parameters))
print(pd.Series(evaluations[family].per_sample[0]).to_string())

## 7. Export for Qiskit/Haiqu

The export contains a valid bound OpenQASM circuit and a JSON template with normalization, feature-to-angle mapping, trained offsets, measurements and the classical output head. Agent 3 can bind the compressed angles and submit the circuit through Haiqu.

In [ ]:
paths = export_qiskit_inference(
    qnn['1q'], ROOT / 'artifacts/circuits/notebook_1q_inference',
    reference_features=datasets['1q'].X[0],
)
paths